# Batch #2 - AUX completion + recipe screen + fresh-split confirmation

Three parts in one session, ordered so a cancel keeps the most valuable work:

1. **AUX completion (split 42)** - `AUX_s45`, `AUX_s46` (aux weight 0.5).
   Completes the matched 5-seed AUX arm against batch-1's A160 seeds.
2. **Recipe screen (split 42)** - five one-flag changes vs the batch-1
   baseline, seeds 42-44, paired with batch-1's same-seed controls:
   `freeze2` (freeze-epochs 2), `blr1e5` (backbone-lr 1e-5),
   `unfreeze2` (unfreeze-layers 2), `pooledv` (`--pooled-v`),
   `noamp` (`--no-amp`).
3. **Fresh-split confirmation (split seed 123)** - `F123_A160_s42-44` vs
   `F123_AUX_s42-44`: does the mixed-ensemble gain reproduce on an unseen
   split?

**Checklist before Run All** (Settings, right sidebar):

1. **Accelerator: GPU T4 x2**
2. **Internet: On** (git clone + pip + model download)
3. **Add Input -> Datasets -> `amnpham/lgbtqia`**

**How long:** ~6-6.5 h (23 runs). A cell that is executing cannot idle-timeout -
leave the session alone, don't press Stop (closing the tab is fine).

**Resume:** each run is skipped when its history CSV already exists, so
re-running a batch cell continues where it stopped (within the same session).
A fresh session starts from scratch.

**Outputs:** per-run histories, val-prob dumps, val arrays for both splits and
summaries; `Try/weights/results_small.zip` is refreshed after each phase. When
it prints `BATCH2 COMPLETE`, just tell me - I pull everything via the API.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'  # aux head, flag-off bit-identical


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data + GPU check

Finds the StereoQueerEval TSVs under `/kaggle/input` (attach `amnpham/lgbtqia`),
sets the data/weights env vars and asserts a GPU is present.


In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)
print('tsvs:', [os.path.basename(h) for h in hits], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx


## 2. Run lists and helpers

Part 1+2 (17 runs, split-seed 42) pair with batch-1 on the identical validation
split, so their prob dumps are directly ensemblable with it. Part 3 (6 runs,
split-seed 123) forms its own pool with suffixed val arrays (`_123`).


In [ ]:
BASE = dict(dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
            backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
            pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
            max_len=160, video_max_len=256, no_amp=False)

# Part 1: complete the AUX arm on split 42 (s45/46, pairs with batch-1 A160).
# Part 2: one-flag recipe screen, seeds 42-44, pairs with batch-1 A160 seeds.
ARMS = [
    ('freeze2',   dict(freeze_epochs=2)),
    ('blr1e5',    dict(backbone_lr=1e-5)),
    ('unfreeze2', dict(unfreeze_layers=2)),
    ('pooledv',   dict(pooled_v=True)),
    ('noamp',     dict(no_amp=True)),
]
RUNS42 = [
    dict(BASE, tag='AUX_s45', seed=45, hate_aux_weight=0.5, split_seed=42),
    dict(BASE, tag='AUX_s46', seed=46, hate_aux_weight=0.5, split_seed=42),
] + [
    dict(BASE, **over, tag=f'{arm}_s{s}', seed=s, split_seed=42)
    for arm, over in ARMS for s in (42, 43, 44)
]

# Part 3: fresh-split confirmation (split seed 123): A160 x3 vs AUX x3.
RUNS123 = [
    dict(BASE, tag=f'F123_A160_s{s}', seed=s, split_seed=123) for s in (42, 43, 44)
] + [
    dict(BASE, tag=f'F123_AUX_s{s}', seed=s, hate_aux_weight=0.5, split_seed=123)
    for s in (42, 43, 44)
]

dev = tx.resolve_device('cuda')
tok = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
_, df_val42 = data_mod.split_by_video(df_all, 0.1, 42)
_, df_val123 = data_mod.split_by_video(df_all, 0.1, 123)
VAL = {42: df_val42, 123: df_val123}
for key, dfv in VAL.items():
    sfx = '' if key == 42 else f'_{key}'
    np.save(f'weights/val_langs{sfx}.npy', dfv['lang'].to_numpy(dtype=str))
    np.save(f'weights/val_videos{sfx}.npy', dfv['yt_title'].to_numpy(dtype=str))
    print(f'val[{key}]: {len(dfv)} rows | by lang:', dfv['lang'].value_counts().to_dict(), flush=True)


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if os.path.exists(hist):
        print(f"-- skip (history exists): {hist}", flush=True)
        return True
    aux = r.get('hate_aux_weight')
    aux_flag = f"--hate-aux-weight {aux} " if aux else ''
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"{'--no-amp ' if r['no_amp'] else ''}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} {aux_flag}"
        f"--out {out} --history {hist}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'== {r["tag"]}: exit {rc} after {(time.time() - t0) / 60:.1f} min', flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {h.loc[bi, 'val_macro_f1']:.4f} | acc {h.loc[bi, 'val_acc']:.4f}", flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if not os.path.exists(hist) or not os.path.exists(out):
        print(f'-- no probs for {r["tag"]}', flush=True)
        return
    sfx = '' if r['split_seed'] == 42 else f"_{r['split_seed']}"
    df_val = VAL[r['split_seed']]
    net = model_xattn.build_model(
        model_xattn.DEFAULT_BACKBONE, unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        hate_aux=r.get('hate_aux_weight', 0) > 0,
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(df_val, tok, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=not r['no_amp'])
    np.save(f"weights/{r['tag']}_val_probs.npy", o['probs'])
    lab_path = f'weights/val_labels{sfx}.npy'
    if not os.path.exists(lab_path):
        np.save(lab_path, o['labels'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload-eval macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    """Bundle the small outputs (no .pt checkpoints) as weights/results_small.zip."""
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned: split-42 runs:', len(RUNS42), '| split-123 runs:', len(RUNS123),
      '| total:', len(RUNS42) + len(RUNS123), flush=True)


## 3. Part 1+2: AUX completion + recipe screen (split 42, ~4.5 h)

Order: AUX 45/46 first (quick, completes the main question), then the five
screen arms.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS42, 1):
    print(f'=== [split42 {i}/{len(RUNS42)}] {r["tag"]} ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
print(f'split-42 phase finished in {(time.time() - t0) / 3600:.2f} h', flush=True)
zip_small()


## 4. Part 3: fresh-split confirmation (split seed 123, ~1.5 h)

Same commands; only the split seed changes. This is the honest test of whether
mixing AUX with A160 reproduces on data the recipe never saw.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS123, 1):
    print(f'=== [fresh123 {i}/{len(RUNS123)}] {r["tag"]} ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
print(f'fresh-split phase finished in {(time.time() - t0) / 3600:.2f} h', flush=True)
zip_small()


## 5. Summary

Per-run table; in-session ensembles (screen arms on split 42; the F123 A160/AUX
pairs on split 123). Cross-batch ensembles with batch-1 are computed offline.


In [ ]:
ALL_RUNS = RUNS42 + RUNS123
try:
    rows = []
    for r in ALL_RUNS:
        hist = f"weights/{r['tag']}_history.csv"
        if os.path.exists(hist):
            h = pd.read_csv(hist)
            bi = h['val_macro_f1'].idxmax()
            rows.append(dict(tag=r['tag'], split=r['split_seed'], seed=r['seed'],
                             epochs=len(h), best_epoch=int(h.loc[bi, 'epoch']),
                             best_f1=round(float(h.loc[bi, 'val_macro_f1']), 4),
                             best_acc=round(float(h.loc[bi, 'val_acc']), 4)))
        else:
            rows.append(dict(tag=r['tag'], split=r['split_seed'], seed=r['seed'],
                             epochs=0, best_epoch=None, best_f1=None, best_acc=None))
    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False), flush=True)
    summary.to_csv('weights/run_summary.csv', index=False)

    def ev(tags, labels):
        ps = [np.load(f"weights/{t}_val_probs.npy") for t in tags
              if os.path.exists(f"weights/{t}_val_probs.npy")]
        if len(ps) < 2:
            return None
        probs = np.mean(ps, axis=0)
        return tx.metrics_from(probs.argmax(1), labels), probs, len(ps)

    ens_rows = []
    if os.path.exists('weights/val_labels.npy'):
        labs42 = np.load('weights/val_labels.npy')
        GROUPS42 = {f'{arm}_x3': [f'{arm}_s{s}' for s in (42, 43, 44)] for arm, _ in ARMS}
        GROUPS42['screenALL_x15'] = [f'{arm}_s{s}' for arm, _ in ARMS for s in (42, 43, 44)]
        for name, tags in GROUPS42.items():
            res = ev(tags, labs42)
            if res is None:
                continue
            m, probs, n = res
            ens_rows.append(dict(ensemble=name, split=42, n=n,
                                 macro_f1=round(float(m['macro_f1']), 4),
                                 acc=round(float(m['acc']), 4)))
            print(f"ENSEMBLE {name}: macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)

    if os.path.exists('weights/val_labels_123.npy'):
        labs123 = np.load('weights/val_labels_123.npy')
        for name, tags in [('F123_A160_x3', [f'F123_A160_s{s}' for s in (42, 43, 44)]),
                           ('F123_AUX_x3', [f'F123_AUX_s{s}' for s in (42, 43, 44)])]:
            res = ev(tags, labs123)
            if res is None:
                continue
            m, probs, n = res
            ens_rows.append(dict(ensemble=name, split=123, n=n,
                                 macro_f1=round(float(m['macro_f1']), 4),
                                 acc=round(float(m['acc']), 4)))
            print(f"ENSEMBLE {name}: macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)
            langs = np.load('weights/val_langs_123.npy')
            for lang in sorted(set(langs)):
                sel = langs == lang
                mm = tx.metrics_from(probs.argmax(1)[sel], labs123[sel])
                print(f"    [{name} {lang}] macro_f1 {mm['macro_f1']:.4f}", flush=True)
    pd.DataFrame(ens_rows).to_csv('weights/ensemble_summary.csv', index=False)
except Exception as e:                                        # noqa: BLE001
    print(f'!! summary failed: {type(e).__name__}: {e}', flush=True)

print('files in weights/:', sorted(os.listdir('weights')), flush=True)
zip_small()
print('BATCH2 COMPLETE - tell the assistant; outputs can be pulled via the API', flush=True)


## Done

Everything needed for the analysis is in `Try/weights/` (histories, prob dumps,
val arrays for both splits, summaries) and bundled as `results_small.zip`.
Just tell me when it prints `BATCH2 COMPLETE`.
